## Lab 4C - Wake Word Detection using ResNet8 Teanfer Learning and  1D CNN

Wake word detection enables voice-activated systems by identifying specific trigger phrases. 

This lab implements binary classification using Google Speech Commands v2 dataset, comparing Res8 and 1D CNN architectures. 

Students will master audio feature extraction (Mel spectrograms + MFCCs), data augmentation techniques, and handle class imbalance using Focal Loss for robust real-world deployment.

Dataset: 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import os
import numpy as np
from sklearn.model_selection import train_test_split
import random


In [ ]:
# ============================================================================
# 1. DATASET CLASS
# ============================================================================
class WakeWordDataset(Dataset):
    def __init__(self, file_paths, labels, sample_rate=16000, n_mels=40, n_mfcc=13, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.n_mfcc = n_mfcc
        self.augment = augment
        
        # Transforms
        self.mel_spec = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=512,
            hop_length=160,
            n_mels=n_mels
        )
        self.mfcc_transform = T.MFCC(
            sample_rate=sample_rate,
            n_mfcc=n_mfcc,
            melkwargs={'n_fft': 512, 'hop_length': 160, 'n_mels': n_mels}
        )
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        # Load audio
        waveform, sr = torchaudio.load(self.file_paths[idx])
        
        # Resample if needed
        if sr != self.sample_rate:
            resampler = T.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)
        
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Fix length to 1 second
        target_length = self.sample_rate
        if waveform.shape[1] < target_length:
            waveform = F.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]
        
        # Data augmentation
        if self.augment:
            waveform = self.augment_audio(waveform)
        
        # Extract features
        mel = self.mel_spec(waveform)  # (1, n_mels, time)
        mfcc = self.mfcc_transform(waveform)  # (1, n_mfcc, time)
        
        # Concatenate mel and mfcc
        features = torch.cat([mel, mfcc], dim=1)  # (1, n_mels+n_mfcc, time)
        
        # Convert to dB scale
        features = torchaudio.transforms.AmplitudeToDB()(features)
        
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return features.squeeze(0), label
    
    def augment_audio(self, waveform):
        """Apply random augmentation"""
        # Time stretch
        if random.random() > 0.5:
            rate = random.uniform(0.8, 1.2)
            waveform = self.time_stretch(waveform, rate)
        
        # Pitch shift
        if random.random() > 0.5:
            n_steps = random.randint(-2, 2)
            waveform = self.pitch_shift(waveform, n_steps)
        
        # Add noise
        if random.random() > 0.5:
            noise = torch.randn_like(waveform) * 0.005
            waveform = waveform + noise
        
        return waveform
    
    def time_stretch(self, waveform, rate):
        """Simple time stretching by resampling"""
        stretched = F.interpolate(
            waveform.unsqueeze(0), 
            size=int(waveform.shape[1] / rate),
            mode='linear',
            align_corners=False
        ).squeeze(0)
        return stretched
    
    def pitch_shift(self, waveform, n_steps):
        """Pitch shift by resampling"""
        rate = 2 ** (n_steps / 12)
        shifted = self.time_stretch(waveform, rate)
        return shifted

In [ ]:
# ============================================================================
# 2. MODEL ARCHITECTURES
# ============================================================================

# Res8 Model (Residual CNN)
class Res8(nn.Module):
    def __init__(self, n_input=53, n_output=1):  # n_input = n_mels + n_mfcc
        super(Res8, self).__init__()
        
        self.conv0 = nn.Conv2d(1, 45, (3, 3), padding=(1, 1), bias=False)
        self.bn0 = nn.BatchNorm2d(45)
        
        # Residual blocks
        self.res_blocks = nn.ModuleList([
            self._make_res_block(45, 45) for _ in range(3)
        ])
        
        self.conv1 = nn.Conv2d(45, 45, (3, 3), padding=(1, 1), bias=False)
        self.bn1 = nn.BatchNorm2d(45)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(45, n_output)
        
    def _make_res_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, (3, 3), padding=(1, 1), bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, (3, 3), padding=(1, 1), bias=False),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension (B, 1, F, T)
        
        x = F.relu(self.bn0(self.conv0(x)))
        
        # Residual blocks
        for block in self.res_blocks:
            residual = x
            x = block(x)
            x = F.relu(x + residual)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x


# 1D CNN Model
class CNN1D(nn.Module):
    def __init__(self, n_input=53, n_output=1):
        super(CNN1D, self).__init__()
        
        self.conv1 = nn.Conv1d(n_input, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)
        
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(2)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(256, n_output)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        # Input: (B, F, T) where F is features, T is time
        
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

In [ ]:
# ============================================================================
# 3. FOCAL LOSS
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

In [ ]:
# ============================================================================
# 4. TRAINING FUNCTION
# ============================================================================
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(features).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        predictions = torch.sigmoid(outputs) > 0.5
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total

In [ ]:
# ============================================================================
# 5. EVALUATION FUNCTION
# ============================================================================
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    tp, fp, tn, fn = 0, 0, 0, 0
    
    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)
            
            outputs = model(features).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            predictions = torch.sigmoid(outputs) > 0.5
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            # Calculate confusion matrix elements
            tp += ((predictions == 1) & (labels == 1)).sum().item()
            fp += ((predictions == 1) & (labels == 0)).sum().item()
            tn += ((predictions == 0) & (labels == 0)).sum().item()
            fn += ((predictions == 0) & (labels == 1)).sum().item()
    
    accuracy = correct / total
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Acceptance Rate
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Rejection Rate
    
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'far': far,
        'frr': frr
    }

In [ ]:
# ============================================================================
# 7. INFERENCE FUNCTION
# ============================================================================
def predict(model, audio_path, device, threshold=0.5):
    """Predict if audio contains wake word"""
    model.eval()
    
    dataset = WakeWordDataset([audio_path], [0], augment=False)
    features, _ = dataset[0]
    features = features.unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(features).squeeze()
        probability = torch.sigmoid(output).item()
    
    is_wake_word = probability > threshold
    
    return is_wake_word, probability

In [ ]:
# ============================================================================
# 6. MAIN TRAINING SCRIPT
# ============================================================================
def main():
    # Hyperparameters
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 10
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # ========================================================================
    # DATA PREPARATION
    # ========================================================================
    # Example: Prepare file paths and labels
    # You need to organize your data as:
    # - Positive samples (wake word): label = 1
    # - Negative samples (other words + silence): label = 0
    
    
    
    data_dir = '/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/speech_commands_v0.02'
    wake_word = 'yes'  # Choose your wake word
    
    positive_files = [os.path.join(data_dir, wake_word, f) 
                      for f in os.listdir(os.path.join(data_dir, wake_word))]
    
    negative_files = []
    for word in ['no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']:
        word_dir = os.path.join(data_dir, word)
        negative_files.extend([os.path.join(word_dir, f) 
                               for f in os.listdir(word_dir)])
    
    # Add background noise
    noise_dir = os.path.join(data_dir, '_background_noise_')
    # Split noise files into 1-second chunks
    
    all_files = positive_files + negative_files
    all_labels = [1] * len(positive_files) + [0] * len(negative_files)
     
    
   
    # For demonstration, assuming data is ready:
    train_files, val_files, train_labels, val_labels = train_test_split(
         all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42
    )
    
    # ========================================================================
    # CREATE DATASETS
    # ========================================================================
    train_dataset = WakeWordDataset(train_files, train_labels, augment=True)
    val_dataset = WakeWordDataset(val_files, val_labels, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    # ========================================================================
    # MODEL INITIALIZATION
    # ========================================================================
    print("\n" + "="*50)
    print("TRAINING RES8 MODEL")
    print("="*50)
    
    model_res8 = Res8(n_input=53, n_output=1).to(DEVICE)
    optimizer_res8 = torch.optim.Adam(model_res8.parameters(), lr=LEARNING_RATE)
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    scheduler_res8 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_res8, mode='min', factor=0.5, patience=5
    )
    
    # Training loop for Res8
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model_res8, train_loader, optimizer_res8, criterion, DEVICE)
        val_metrics = evaluate(model_res8, val_loader, criterion, DEVICE)
         
        scheduler_res8.step(val_metrics['loss'])
        
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")
        print(f"FAR: {val_metrics['far']:.4f}, FRR: {val_metrics['frr']:.4f}")
        print(f"Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        print("-" * 50)
    
    print("\n" + "="*50)
    print("TRAINING 1D CNN MODEL")
    print("="*50)
    
    model_cnn1d = CNN1D(n_input=53, n_output=1).to(DEVICE)
    optimizer_cnn1d = torch.optim.Adam(model_cnn1d.parameters(), lr=LEARNING_RATE)
    scheduler_cnn1d = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn1d, mode='min', factor=0.5, patience=5
    )
    
    # Training loop for 1D CNN
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model_cnn1d, train_loader, optimizer_cnn1d, criterion, DEVICE)
        val_metrics = evaluate(model_cnn1d, val_loader, criterion, DEVICE)
        
        scheduler_cnn1d.step(val_metrics['loss'])
        
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")
        print(f"FAR: {val_metrics['far']:.4f}, FRR: {val_metrics['frr']:.4f}")
        print(f"Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        print("-" * 50)
    
    # Save models
    torch.save(model_res8.state_dict(), 'wake_word_res8.pth')
    torch.save(model_cnn1d.state_dict(), 'wake_word_cnn1d.pth')
    





if __name__ == "__main__":
    main()